## NF-CORE/MAG setup for "full" pipeline and "assembled" partial execution 

There are two ways to run the nf-core/mag workflow: either starting from the raw (adaptor-removed and cleaned) sequences (the same ones we get back from Genoscope), or starting from assembled sequences (the MEGAHIT final.contigs.fa sequences from MGF). If using the "assembled" partial workflow, then the initial cleaning, trimming, and assembly steps of the full workflow are skipped.

### Setup

```
.
├── custom.config
├── data
│   ├── HCFCYDSX5.UDI129
│   ├── HCFCYDSX5.UDI135
│   └── HJWK3DSX7.UDI362
├── libs
│   ├── bacteria_odb10.2024-01-08.tar.gz
│   └── gtdbtk_r220_data
├── results
│   ├── assembly
│   └── full
├── prepare-nf-core-mag-full.sh
├── prepare-nf-core-mag-assembled.sh
├── UDI129ma.log
├── UDI129ma-sbatch.sh
└── work

```

- `custom.config` - workflow parameters overriding defaults
- `/data` - data directory
- `/libs` - other cached libraries (may or may not be used)
- `/results`
- `/results/assembled` - where the results of the "assembled" wf go
- `/results/full` - where the results of the "full" workflow go
- `/work`- wf directory where temporary directories are place (clean periodically)

**NB as of 06/06/2025 because the db is downloaded each time the wf runs and keeps failing. So pre-download the database, untar/compress it and place in /libs
wget https://data.ace.uq.edu.au/public/gtdb/data/releases/release220/220.0/auxillary_files/gtdbtk_package/full_package/gtdbtk_r220_data.tar.gz**

##### Tags

```
UDI129ma - assembled wf
```

```
UDI129mf - full wf
```

### Data files

##### Full worklow
Only requires the raw sequences from Genoscope:
```
redi:/usr/local/scratch/emo-bon-sequencing-data/www.genoscope.cns.fr/sadc/projet_DBB
```

##### Assembled workflow
Requires the same two raw sequence data files from Genoscope, plus the assembled contigs from MGF. The assembled contigs from MGF are all named the same in very MGF results: final.contig.fa

**NB the contigs need to be gzipped**

```
$ gzip -c final.contigs.fa > final.contigs.fa.gz
```


### Custom parameters for running assembled workflow starting from MEGAHIT contig of MGF (will run on any node with 110GB RAM)
```
$ cat custom.config 
process {
    resourceLimits = [
        cpus: 54,
        memory: 110.GB,
        time: 96.h
    ]
    withName: METASPADES {
        cpus = 54
        memory = 110.GB
    }
    withName: MEGAHIT {
        time = 48.h
    }
    withName: BOWTIE2_ASSEMBLY_ALIGN {
        time = 48.h
    }
}
```

### Custom parameters for running full workflow (including metaspaces - must run on ceta-gen - needs > 250GB RAM)
```
$ cat custom-bigmem.config 
process {
    resourceLimits = [
        cpus: 54,
        memory: 800.GB,
        time: 96.h
    ]
    withName: METASPADES {
        cpus = 54
        memory = 800.GB
        time = 96.h
    }
    withName: MEGAHIT {
        time = 48.h
    }
    withName: BOWTIE2_ASSEMBLY_ALIGN {
        time = 48.h
    }
}
```

### Full workflow using METASPADES on ceta-gen (big-mem)


```
$ cat prepare-nf-core-mag-full.sh 
#!/bin/bash

DATA_FORWARD=$1
NUM=`expr substr $DATA_FORWARD 14 1`
if [[ "$NUM" = 1 ]]; then
    DATA_REVERSE=${DATA_FORWARD/_1_1_/_1_2_}
elif [[ "$NUM" = 2 ]]; then
    DATA_REVERSE=${DATA_FORWARD/_2_1_/_2_2_}
elif [[ "$NUM" = 3 ]]; then
    DATA_REVERSE=${DATA_FORWARD/_3_1_/_3_2_}
fi
RUN_CODE=${DATA_FORWARD:17:16}
UDI=${RUN_CODE:10:6}
SUBFILE=${UDI}mf-sbatch.sh
INPUT_FILEPATH="data/${RUN_CODE}/input.csv"

echo "DATA_FORWARD = $DATA_FORWARD"
echo "DATA_REVERSE = $DATA_REVERSE"
echo "RUN_CODE = $RUN_CODE"
echo "UDI = $UDI"
echo "SUBFILE = $SUBFILE"
echo "INPUT_FILEPATH = $INPUT_FILEPATH"

cat > $SUBFILE <<EOF
#!/bin/bash

#SBATCH --nodes=1
#SBATCH --nodelist=ceta-gen
#SBATCH --partition=bigmem
#SBATCH --ntasks-per-node=1
#SBATCH --cpus-per-task=54
#SBATCH --job-name=${UDI}mf
#SBATCH --output=${UDI}mf.log
#SBATCH --error=${UDI}mf.err

export JAVA_HOME=/usr/lib/jvm/java-17-openjdk-17.0.15.0.6-2.el8.x86_64
export PATH=\$JAVA_HOME/bin:\$PATH
export NXF_APPTAINER_CACHEDIR=/share/apps/share/nextflow/apptainer_cache

date
nextflow -log results/full/${RUN_CODE}/nexflow.log \\
    run nf-core/mag -r 4.0.0 \\
    -c custom-bigmem.config \\
    -profile apptainer \\
    --input data/${RUN_CODE}/input.csv \\
    --outdir results/full/${RUN_CODE} \\
    --gtdb_db libs/gtdbtk_release220 \\
    --busco_db libs/bacteria_odb10.2024-01-08.tar.gz \\
    --skip_megahit \\
    --skip_concoct \\
    --skip_metabat2

#    -resume
#    --skip_gtdbtk \\
date
EOF

cat > $INPUT_FILEPATH <<EOF
sample,group,short_reads_1,short_reads_2,long_reads
${UDI},0,data/${RUN_CODE}/${DATA_FORWARD},data/${RUN_CODE}/${DATA_REVERSE},
EOF

echo
echo "Input file:"
cat $INPUT_FILEPATH
echo
echo "Submission script:"
cat $SUBFILE
echo
echo
echo "This nf-core/mag wf does contig assembly using METASPADES and binning using MAXBIN2 only running on ceta-gen!"

```

#### How to use:
```
$ prepare-nf-core-mag-full.sh DBH_AABAOSDA_1_1_HCFCYDSX5.UDI129_clean.fastq.gz
$ sbatch UDI129mf-sbatch.sh
```

### Assembled workflow

**This nf-core/mag wf uses pre-computed MEGAHIT contigs and does binning using MAXBIN2 only**

```
$ cat prepare-nf-core-mag-assembled.sh
#!/bin/bash

DATA_FORWARD=$1
NUM=`expr substr $DATA_FORWARD 14 1`
if [[ "$NUM" = 1 ]]; then
    DATA_REVERSE=${DATA_FORWARD/_1_1_/_1_2_}
elif [[ "$NUM" = 2 ]]; then
    DATA_REVERSE=${DATA_FORWARD/_2_1_/_2_2_}
elif [[ "$NUM" = 3 ]]; then
    DATA_REVERSE=${DATA_FORWARD/_3_1_/_3_2_}
fi
RUN_CODE=${DATA_FORWARD:17:16}
UDI=${RUN_CODE:10:6}
SUBFILE=${UDI}ma-sbatch.sh
ASS_INPUT_FILEPATH="data/${RUN_CODE}/assembled_input.csv"
INPUT_FILEPATH="data/${RUN_CODE}/input.csv"

echo "DATA_FORWARD = $DATA_FORWARD"
echo "DATA_REVERSE = $DATA_REVERSE"
echo "RUN_CODE = $RUN_CODE"
echo "UDI = $UDI"
echo "SUBFILE = $SUBFILE"
echo "ASS_INPUT_FILEPATH = $ASS_INPUT_FILEPATH"
echo "INPUT_FILEPATH = $INPUT_FILEPATH"

cat > $SUBFILE <<EOF
#!/bin/bash

##SBATCH --exclude=ceta3,ceta4,ceta5,ceta6,ceta-gen,ceta.ualg.pt
#SBATCH --nodes=1
#SBATCH --ntasks-per-node=1
#SBATCH --cpus-per-task=54
#SBATCH --job-name=${UDI}ma
#SBATCH --output=${UDI}ma.log
#SBATCH --error=${UDI}ma.err

export JAVA_HOME=/usr/lib/jvm/java-17-openjdk-17.0.15.0.6-2.el8.x86_64
export PATH=\$JAVA_HOME/bin:\$PATH
export NXF_APPTAINER_CACHEDIR=/share/apps/share/nextflow/apptainer_cache

date
nextflow -log results/assembled/${RUN_CODE}/nexflow.log \\
    run nf-core/mag -r 4.0.0 \\
    -c custom.config \\
    -profile apptainer \\
    --assembly_input data/${RUN_CODE}/assembled_input.csv \\
    --input data/${RUN_CODE}/input.csv \\
    --outdir results/assembled/${RUN_CODE} \\
    --gtdb_db libs/gtdbtk_release220 \\
    --busco_db libs/bacteria_odb10.2024-01-08.tar.gz \\
    --skip_concoct \\
    --skip_metabat2

#    --skip_gtdbtk
#    -resume
date
EOF

cat > $ASS_INPUT_FILEPATH <<EOF
id,group,assembler,fasta
${UDI},0,MEGAHIT,data/${RUN_CODE}/final.contigs.fa.gz
EOF

cat > $INPUT_FILEPATH <<EOF
sample,group,short_reads_1,short_reads_2,long_reads
${UDI},0,data/${RUN_CODE}/${DATA_FORWARD},data/${RUN_CODE}/${DATA_REVERSE},
EOF

echo
echo "Input file:"
cat $INPUT_FILEPATH
echo
echo "Assembly input file:"
cat $ASS_INPUT_FILEPATH
echo 
echo "Submission script:"
cat $SUBFILE
echo
echo
echo "This nf-core/mag wf uses pre-computed MEGAHIT contigs and does binning using MAXBIN2 only"

```

#### How to use:
```
$ prepare-nf-core-mag-assembled.sh DBH_AABAOSDA_1_1_HCFCYDSX5.UDI129_clean.fastq.gz
$ sbatch UDI129mf-sbatch.sh
```
